In [ ]:
# =============================================================================
# PRESS TRUST OF INDIA (PTI) FACT CHECK ARCHIVE SCRAPER
# =============================================================================


# -----------------------------------------------------------------------------
# 1. INSTALL DEPENDENCIES
# -----------------------------------------------------------------------------
%pip -q install requests beautifulsoup4 pandas tqdm lxml


In [ ]:

# -----------------------------------------------------------------------------
# 2. IMPORTS
# -----------------------------------------------------------------------------
import json
import re
import time
from html import unescape
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

from google.colab import drive


In [ ]:
# -----------------------------------------------------------------------------
# 3. MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------
# Change the folder name accordingly
MOUNT_POINT = "/content/gdrive"

drive.mount(MOUNT_POINT)

OUTPUT_DIR = Path(MOUNT_POINT) / "MyDrive" / "PTI_Fact_Check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Persistent outputs
CHECKPOINT_CSV = OUTPUT_DIR / "pti_archive_checkpoint.csv"
PROGRESS_JSON = OUTPUT_DIR / "pti_archive_progress.json"
FAILED_CSV = OUTPUT_DIR / "pti_archive_failed_offsets.csv"
ALL_RECORDS_CSV = OUTPUT_DIR / "pti_fact_check_archive_all.csv"
FILTERED_CSV = OUTPUT_DIR / "pti_fact_check_archive_2022_2025.csv"

In [ ]:
# -----------------------------------------------------------------------------
# 4. SETTINGS
# -----------------------------------------------------------------------------
API_BASE = "https://apii.ptivideos.com/pti"
ARCHIVE_ENDPOINT = f"{API_BASE}/archival-fact-check"
REFERER = "https://www.ptinews.com/fact-check"

BATCH_SIZE = 5
PAUSE_SECONDS = 1.5
REQUEST_TIMEOUT = (20, 120)
MAX_RETRIES_PER_BATCH = 3

# A checkpoint is saved after this many successfully handled batches.
CHECKPOINT_EVERY_BATCHES = 10

START_YEAR = 2022
END_YEAR = 2025

# The API returned 2,055 records in the inspection performed for this scraper.
# The script first asks the endpoint for its current count and uses that value.
# If the count cannot be read, this value is used as a fallback.
FALLBACK_TOTAL_RECORDS = 2055

In [ ]:

# -----------------------------------------------------------------------------
# 5. HTTP SESSION WITH AUTOMATIC RETRIES
# -----------------------------------------------------------------------------
session = requests.Session()

retry_policy = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)

adapter = HTTPAdapter(max_retries=retry_policy)
session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": REFERER,
    "Origin": "https://www.ptinews.com",
})



In [ ]:
# -----------------------------------------------------------------------------
# 6. CLEANING AND DATE HELPERS
# -----------------------------------------------------------------------------
def clean_html(value):
    """
    Convert HTML strings to readable text.
    Lists and dictionaries are handled recursively; None becomes an empty
    string. Image, script, style, iframe, and noscript elements are removed.
    """
    if value is None:
        return ""

    if isinstance(value, list):
        return " ".join(
            clean_html(item) for item in value if item is not None
        ).strip()

    if isinstance(value, dict):
        return " ".join(
            clean_html(item) for item in value.values() if item is not None
        ).strip()

    text = unescape(str(value))
    if not text.strip():
        return ""

    soup = BeautifulSoup(text, "lxml")

    for tag in soup(["img", "script", "style", "noscript", "iframe"]):
        tag.decompose()

    return " ".join(soup.get_text(" ", strip=True).split()).strip()


def extract_author(article_text):
    """
    Try to extract the author/byline from the opening story text.
    Example format:
    'New Delhi, September 7 (Name/Name, PTI Fact Check): ...'
    """
    if not article_text:
        return ""

    pattern = re.compile(
        r"\(([^()]*(?:PTI Fact[- ]Check|PTI Fact Check Desk)[^()]*)\)",
        flags=re.IGNORECASE,
    )
    match = pattern.search(article_text)

    if not match:
        return ""

    byline = match.group(1)
    byline = re.sub(
        r",?\s*PTI Fact[- ]Check(?: Desk)?",
        "",
        byline,
        flags=re.IGNORECASE,
    )
    return byline.strip(" ,/")


def parse_year(value):
    """Return a year from a date-like string, or None when unavailable."""
    if value is None:
        return None

    text = str(value).strip()
    match = re.search(r"\b(19\d{2}|20\d{2})\b", text)
    return int(match.group(1)) if match else None


def safe_write_csv(dataframe, filepath):
    """Write atomically to reduce the risk of a partially written checkpoint."""
    filepath = Path(filepath)
    temporary = filepath.with_suffix(filepath.suffix + ".tmp")
    dataframe.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(filepath)


In [ ]:
# -----------------------------------------------------------------------------
# 7. GET THE CURRENT ARCHIVE COUNT
# -----------------------------------------------------------------------------
def get_archive_count():
    """Ask the API for its current total count; use the configured fallback."""
    try:
        response = session.get(
            ARCHIVE_ENDPOINT,
            params={"start": 0, "length": 1},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        payload = response.json()

        count = payload.get("count") if isinstance(payload, dict) else None
        if count is not None:
            return int(count)

        print("API response did not include a count; using fallback.")
    except (requests.RequestException, ValueError, TypeError) as exc:
        print(f"Could not obtain archive count: {exc}")
        print("Using the configured fallback archive count.")

    return FALLBACK_TOTAL_RECORDS


TOTAL_RECORDS = get_archive_count()
print("Archive records reported:", TOTAL_RECORDS)


In [ ]:
# -----------------------------------------------------------------------------
# 8. FETCH A SINGLE BATCH
# -----------------------------------------------------------------------------
def fetch_batch(offset, length=BATCH_SIZE):
    """
    Fetch one API page.

    Raises an exception if the request fails or the response structure is
    unexpected. A failed batch is not marked complete, so it can be retried.
    """
    response = session.get(
        ARCHIVE_ENDPOINT,
        params={"start": offset, "length": length},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, dict):
        raise ValueError(f"Unexpected JSON response at offset {offset}")

    batch = payload.get("archival_fact_check", [])
    if batch is None:
        batch = []

    if not isinstance(batch, list):
        raise ValueError(f"Unexpected archive data structure at offset {offset}")

    return batch

In [ ]:
# -----------------------------------------------------------------------------
# 9. CONVERT AN API RECORD INTO A CLEAN ROW
# -----------------------------------------------------------------------------
def clean_record(item):
    """Clean a raw API record while retaining useful original fields."""
    story_text = clean_html(item.get("story", ""))
    headline = clean_html(item.get("headline", ""))

    return {
        "id": str(item.get("id", "")).strip(),
        "title": headline,
        "author": extract_author(story_text),
        "date": clean_html(item.get("date", "")),
        "releasetime": clean_html(item.get("releasetime", "")),
        "created_at": clean_html(item.get("created_at", "")),
        "origin": clean_html(item.get("origin", "")),
        "source": clean_html(item.get("source", "")),
        "category": clean_html(item.get("category", "")),
        "claim": clean_html(item.get("claim", "")),
        "fact": clean_html(item.get("fact", "")),
        "conclusion": clean_html(item.get("conclusion", "")),
        "feedback": clean_html(item.get("feedback", "")),
        "article_text": story_text,
        "textstory": clean_html(item.get("textstory", "")),
        # Keep image fields as provided by the API; they may be URLs or values.
        "image": item.get("image"),
        "newimage": item.get("newimage"),
    }


OUTPUT_COLUMNS = [
    "id", "title", "author", "date", "releasetime", "created_at",
    "origin", "source", "category", "claim", "fact", "conclusion",
    "feedback", "article_text", "textstory", "image", "newimage",
]



In [ ]:

# -----------------------------------------------------------------------------
# 10. LOAD PREVIOUS CHECKPOINT AND PROGRESS
# -----------------------------------------------------------------------------
records_by_id = {}

if CHECKPOINT_CSV.exists():
    previous_df = pd.read_csv(CHECKPOINT_CSV, dtype=str).fillna("")
    if "id" in previous_df.columns:
        for row in previous_df.to_dict("records"):
            record_id = str(row.get("id", "")).strip()
            if record_id and record_id.lower() != "nan":
                records_by_id[record_id] = row
    print("Checkpoint records loaded:", len(records_by_id))
else:
    print("No checkpoint CSV found; starting with no saved records.")

if PROGRESS_JSON.exists():
    try:
        with open(PROGRESS_JSON, "r", encoding="utf-8") as file:
            progress = json.load(file)
        completed_offsets = {
            int(offset) for offset in progress.get("completed_offsets", [])
        }
    except (json.JSONDecodeError, OSError, TypeError, ValueError):
        completed_offsets = set()
        print("Progress file could not be read; rebuilding progress state.")
else:
    completed_offsets = set()

if FAILED_CSV.exists():
    try:
        failed_df = pd.read_csv(FAILED_CSV)
        failed_offsets = set(
            failed_df["failed_offset"].dropna().astype(int).tolist()
        ) if "failed_offset" in failed_df.columns else set()
    except Exception:
        failed_offsets = set()
else:
    failed_offsets = set()

print("Completed batches loaded:", len(completed_offsets))
print("Previously failed offsets loaded:", len(failed_offsets))
print("Unique record IDs in checkpoint:", len(records_by_id))



In [ ]:
# -----------------------------------------------------------------------------
# 11. CHECKPOINT FUNCTION
# -----------------------------------------------------------------------------
def save_checkpoint():
    """Persist records, completed offsets, and failed offsets to Drive."""
    checkpoint_df = pd.DataFrame(
        list(records_by_id.values()),
        columns=OUTPUT_COLUMNS,
    )
    safe_write_csv(checkpoint_df, CHECKPOINT_CSV)

    progress_payload = {
        "total_records_at_start": TOTAL_RECORDS,
        "batch_size": BATCH_SIZE,
        "completed_offsets": sorted(completed_offsets),
        "failed_offsets": sorted(failed_offsets),
    }

    temporary_json = PROGRESS_JSON.with_suffix(".json.tmp")
    with open(temporary_json, "w", encoding="utf-8") as file:
        json.dump(progress_payload, file, indent=2)
    temporary_json.replace(PROGRESS_JSON)

    safe_write_csv(
        pd.DataFrame({"failed_offset": sorted(failed_offsets)}),
        FAILED_CSV,
    )


In [ ]:
# -----------------------------------------------------------------------------
# 12. DETERMINE WHICH BATCHES NEED PROCESSING
# -----------------------------------------------------------------------------
all_offsets = list(range(0, TOTAL_RECORDS, BATCH_SIZE))

# Failed offsets are explicitly included even if an earlier progress file
# accidentally marked them complete.
pending_offsets = [
    offset for offset in all_offsets
    if offset not in completed_offsets or offset in failed_offsets
]

print("\nTotal archive batches:", len(all_offsets))
print("Batches already completed:", len(completed_offsets))
print("Batches remaining/requiring retry:", len(pending_offsets))


In [ ]:

# -----------------------------------------------------------------------------
# 13. SCRAPE BATCHES WITH CHECKPOINTING
# -----------------------------------------------------------------------------
batches_since_checkpoint = 0
new_records_this_run = 0

for offset in tqdm(pending_offsets, desc="Scraping PTI Fact Check archive"):
    succeeded = False

    for attempt in range(1, MAX_RETRIES_PER_BATCH + 1):
        try:
            batch = fetch_batch(offset, BATCH_SIZE)

            for item in batch:
                if not isinstance(item, dict):
                    continue

                cleaned = clean_record(item)
                record_id = cleaned["id"]

                if not record_id or record_id.lower() == "nan":
                    continue

                # Insert or refresh this ID. Repeated API records are deduped.
                if record_id not in records_by_id:
                    new_records_this_run += 1
                records_by_id[record_id] = cleaned

            # Only mark complete after a valid response was handled.
            completed_offsets.add(offset)
            failed_offsets.discard(offset)
            succeeded = True
            break

        except Exception as exc:
            print(
                f"\nOffset {offset}, attempt {attempt}/"
                f"{MAX_RETRIES_PER_BATCH} failed: {repr(exc)}"
            )
            time.sleep(2 * attempt)

    if not succeeded:
        failed_offsets.add(offset)

    batches_since_checkpoint += 1

    if batches_since_checkpoint >= CHECKPOINT_EVERY_BATCHES:
        save_checkpoint()
        batches_since_checkpoint = 0
        print(
            f"\nCheckpoint saved | unique records: {len(records_by_id)} | "
            f"completed batches: {len(completed_offsets)} | "
            f"failed batches: {len(failed_offsets)}"
        )

    time.sleep(PAUSE_SECONDS)

# Save all progress at the end of the first pass.
save_checkpoint()

In [ ]:

# -----------------------------------------------------------------------------
# 14. RETRY FAILED OFFSETS ONCE MORE
# -----------------------------------------------------------------------------
if failed_offsets:
    print("\nRetrying batches that remain failed...")

    for offset in tqdm(sorted(failed_offsets), desc="Retrying failed batches"):
        try:
            batch = fetch_batch(offset, BATCH_SIZE)

            for item in batch:
                if not isinstance(item, dict):
                    continue

                cleaned = clean_record(item)
                record_id = cleaned["id"]

                if not record_id or record_id.lower() == "nan":
                    continue

                if record_id not in records_by_id:
                    new_records_this_run += 1
                records_by_id[record_id] = cleaned

            completed_offsets.add(offset)
            failed_offsets.discard(offset)

        except Exception as exc:
            print(f"\nRetry still failed at offset {offset}: {repr(exc)}")
            failed_offsets.add(offset)

        # Save after each retry so a runtime interruption loses little work.
        save_checkpoint()
        time.sleep(PAUSE_SECONDS)


In [ ]:
# -----------------------------------------------------------------------------
# 15. SAVE FINAL COMPLETE DATASET
# -----------------------------------------------------------------------------
full_df = pd.DataFrame(
    list(records_by_id.values()),
    columns=OUTPUT_COLUMNS,
)

if not full_df.empty:
    full_df["id"] = full_df["id"].astype(str).str.strip()
    full_df = full_df[full_df["id"].ne("")]
    full_df = full_df.drop_duplicates(subset="id", keep="last")

    # Sort by date text for easier inspection; raw date field is retained.
    full_df = full_df.sort_values("date", na_position="last")

safe_write_csv(full_df, ALL_RECORDS_CSV)

In [ ]:
# -----------------------------------------------------------------------------
# 16. FILTER RECORDS TO 2022–2025
# -----------------------------------------------------------------------------
# PTI records provide dates in different formats. Extracting the four-digit year
full_df["year"] = full_df["date"].apply(parse_year)

filtered_df = full_df[
    full_df["year"].between(START_YEAR, END_YEAR)
].copy()

filtered_df = filtered_df.drop(columns=["year"], errors="ignore")
safe_write_csv(filtered_df, FILTERED_CSV)





In [ ]:
# -----------------------------------------------------------------------------
# 17. FINAL SUMMARY
# -----------------------------------------------------------------------------
print("\n" + "=" * 72)
print("PTI FACT CHECK SCRAPING SUMMARY")
print("=" * 72)
print("API-reported total:", TOTAL_RECORDS)
print("Unique IDs collected:", len(full_df))
print("New IDs added during this run:", new_records_this_run)
print("Completed batches:", len(completed_offsets))
print("Failed batches remaining:", len(failed_offsets))
print(f"Records dated {START_YEAR}–{END_YEAR}:", len(filtered_df))

print("\nRecords by year:")
if not full_df.empty:
    print(
        full_df["date"].apply(parse_year)
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
    )

print("\nSaved files:")
print("Checkpoint:", CHECKPOINT_CSV)
print("Progress state:", PROGRESS_JSON)
print("Failed offsets:", FAILED_CSV)
print("Complete archive:", ALL_RECORDS_CSV)
print("Filtered 2022–2025 archive:", FILTERED_CSV)

if failed_offsets:
    print(
        "\nSome offsets remain unsuccessful. They are listed in the failed "
        "offsets CSV and will be retried when the script is run again."
    )
else:
    print("\nAll archive offsets were processed successfully.")

print("=" * 72)
display(filtered_df.head(10))

# IMPORTANT RESUME NOTE:
# Run this same script again to resume. It reads the checkpoint and progress
# files from Drive and skips completed offsets. Keep OUTPUT_DIR unchanged.